# Dissertation Experiments: Activation Function Exploration

This notebook contains the training and evaluation workflow for UCI-HAPT and WISDM experiments.

Note: outputs are cleared in version control for a clean, reproducible notebook.


In [ ]:
!pip install tensorflow-macos


In [ ]:
!pip install tensorflow-metal

In [ ]:
!gdown --id 1ECKVBGoWpBYARHMzG7XOW3iXVm_VsDmU -O  dataset_dissertation.zip


In [ ]:
!unzip dataset_dissertation.zip

In [ ]:
import tensorflow as tf


In [ ]:
import numpy as np
import pandas as pd
import tracemalloc
import time
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, precision_score , recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    Dense, Dropout, LSTM, BatchNormalization, PReLU, Layer,
    Activation, Flatten, Concatenate
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

In [ ]:

gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available:", len(gpus))
print(gpus)


In [ ]:
tf.random.set_seed(17)

## **UCI HAPT**

## **Train**

In [ ]:
data_dir_raw = 'Train'
for i in os.listdir(data_dir_raw):
  print(i)

In [ ]:
X_train = pd.read_csv(os.path.join('Train', 'X_train.txt'), sep=r'\s+', header=None)
y_train = pd.read_csv(os.path.join('Train', 'y_train.txt'), sep=r'\s+', header=None)

display(X_train.head())
display(y_train.head())

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
X_val.shape

In [ ]:
y_val.shape

## **Test**

In [ ]:
data_dir_test_raw = 'Test'

In [ ]:
X_test = pd.read_csv(os.path.join('Test', 'X_test.txt'), sep=r'\s+', header=None)
y_test = pd.read_csv(os.path.join('Test', 'y_test.txt'), sep=r'\s+', header=None)

In [ ]:
X_test = np.array(X_test)
y_test = np.array(y_test)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

In [ ]:
X_test.shape


### **Y_TRAIN AND Y_TEST AND Y_VAL**

In [ ]:
y_train = to_categorical(y_train - 1, num_classes=12)
y_val   = to_categorical(y_val - 1, num_classes=12)
y_test  = to_categorical(y_test - 1, num_classes=12)

## **MODELS**

### **ACTIVATIONS**

In [ ]:
class TwoLaneActivation(Layer):
    def __init__(self, **kwargs):
        super(TwoLaneActivation, self).__init__(**kwargs)

    def call(self, inputs):
        positive_output = tf.nn.relu(inputs)  # max(0, x)
        negative_output = tf.minimum(0.0, inputs) # min(0, x)
        return [positive_output, negative_output]

In [ ]:
class Sqrtfunct(tf.keras.layers.Layer):
        def __init__(self, **kwargs):
            super(Sqrtfunct, self).__init__(**kwargs)
        def call(self, inputs):
             abs_x = tf.abs(inputs)
             transformed = inputs + abs_x
             transformed = tf.add(transformed, tf.math.log1p(abs_x))
             return tf.where(inputs > 0, tf.sqrt(transformed), transformed)

In [ ]:
@tf.keras.utils.register_keras_serializable(package='Custom')
class SqrtfunctNonlinear_4(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    @tf.function()
    def call(self, x):
        abs_x = tf.abs(x)
        x2 = abs_x * abs_x
        x3 = x2 * abs_x

        log_approx = abs_x - 0.5 * x2 + (1.0 / 3.0) * x3
        t = x + abs_x + log_approx

        # THE FINAL FIVE — empirically proven best
        y = t * 0.5 + 1.0
        y = 0.5 * (y + t / y)  # 1
        y = 0.5 * (y + t / y)  # 2
        y = 0.5 * (y + t / y)  # 3
        y = 0.5 * (y + t / y)  # 4

        return tf.where(x > 0, y, t)

In [ ]:
@tf.keras.utils.register_keras_serializable(package='Custom')
class SqrtfunctNonlinear_3(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    @tf.function()
    def call(self, x):
        abs_x = tf.abs(x)
        x2 = abs_x * abs_x
        x3 = x2 * abs_x

        log_approx = abs_x - 0.5 * x2 + (1.0 / 3.0) * x3
        t = x + abs_x + log_approx

        # THE FINAL FIVE — empirically proven best
        y = t * 0.5 + 1.0
        y = 0.5 * (y + t / y)  # 1
        y = 0.5 * (y + t / y)  # 2
        y = 0.5 * (y + t / y)  # 3

        return tf.where(x > 0, y, t)

In [ ]:
class StabilizedLog(tf.keras.layers.Layer):
        def __init__(self, **kwargs):
            super(StabilizedLog, self).__init__(**kwargs)

        def call(self, inputs):
            abs_x = tf.abs(inputs)
            inner = inputs + abs_x + 1
            return tf.math.log(inner)

In [ ]:
# --------------------- CReLU CLASS (NEW) ---------------------
class CReLU(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(CReLU, self).__init__(**kwargs)

    def call(self, inputs):
        pos = tf.nn.relu(inputs)
        neg = tf.nn.relu(-inputs)
        return tf.concat([pos, neg], axis=-1)

    def get_config(self):
        return super(CReLU, self).get_config()

#### **TWO WAY**

In [ ]:
def build_model_1():
    inputs = tf.keras.Input(shape=(561,1))

    conv1 = tf.keras.layers.Conv1D(32, 3, padding='same')(inputs)
    pos1, neg1 = TwoLaneActivation()(conv1)
    x = tf.keras.layers.Concatenate()([pos1, neg1])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = tf.keras.layers.Conv1D(64, 3, padding='same')(x)
    pos2, neg2 = TwoLaneActivation()(conv2)
    x = tf.keras.layers.Concatenate()([pos2, neg2])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
build_model_1().summary()

#### **RELU - CNN**

In [ ]:
def build_model_2():
    model = Sequential([
        Conv1D(kernel_size=3, filters=32, padding="same", activation="relu", input_shape=(561,1)),
        BatchNormalization(),
        MaxPooling1D(pool_size=2, padding="same"),

        Conv1D(kernel_size=3, filters=64, padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2, padding="same"),

        Flatten(),
        Dense(512, activation="relu"),
        Dropout(0.5),
        Dense(12, activation="softmax")
    ])

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


In [ ]:
build_model_2().summary()

#### **Square root - Log activation - CNN**

In [ ]:
def build_model_3():
    inputs = tf.keras.Input(shape=(561,1))

    conv1 = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = Sqrtfunct()(conv1)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = Sqrtfunct()(conv2)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = Sqrtfunct()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
build_model_3().summary()

#### **Convolutional model with custom square root approximation using Newton-Raphson method and log approximation using 3 term taylor series**

In [ ]:
from tensorflow.keras.optimizers import legacy
def build_model_nonlin_optimized_4():
    inputs = tf.keras.Input(shape=(561,1))


    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
build_model_nonlin_optimized_4().summary()

In [ ]:
from tensorflow.keras.optimizers import legacy
def build_model_nonlin_optimized_3():
    inputs = tf.keras.Input(shape=(561,1))


    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
build_model_nonlin_optimized_3().summary()

#### **Stabilized Log - CNN**

In [ ]:
def build_model_7():

    inputs = Input(shape=(561,1))

    conv1 = Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = StabilizedLog()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = StabilizedLog()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = StabilizedLog()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model


In [ ]:
build_model_7().summary()

### **Tanh Variation**

In [ ]:
def build_model_tanh():
    inputs = Input(shape=(561,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = tf.math.tanh(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.math.tanh(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.math.tanh)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
build_model_tanh().summary()

### **CRELU Variation**

In [ ]:
def build_model_crelu():
    inputs = Input(shape=(561,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = tf.nn.crelu(conv1)  # concatenates ReLU(x) and ReLU(-x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.nn.crelu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = tf.nn.crelu(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
build_model_crelu().summary()

### **PRELU Variation**

In [ ]:
from tensorflow.keras.layers import PReLU

def build_model_prelu():
    inputs = Input(shape=(561,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = PReLU()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = PReLU()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = PReLU()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
build_model_prelu().summary()

### **ELU Variation**

In [ ]:
def build_model_elu():
    inputs = Input(shape=(561,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = tf.keras.activations.elu(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.keras.activations.elu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.keras.activations.elu)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
build_model_elu().summary()

#### **LSTM**

In [ ]:
from tensorflow.keras.layers import Attention
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_model_lstm():
    inputs = Input(shape=(561,1))

    # LSTM with sequences
    x = LSTM(128, return_sequences=True)(inputs)
    x = BatchNormalization()(x)

    x = LSTM(128, return_sequences=True)(x)
    x = BatchNormalization()(x)

    # Attention: query and value = same sequence (self-attention)
    attention_output = Attention()([x, x])

    # Pooling after attention
    x = tf.reduce_mean(attention_output, axis=1)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model


In [ ]:
build_model_lstm().summary()

##### **Call backs**

In [ ]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_accuracy',           # Watch validation accuracy
    patience=5,                       # Wait 5 epochs for improvement
    factor=0.5,                       # Halve LR when no progress after activity
    min_lr=1e-7,                      # Minimum LR floor
    verbose=1,
    mode='max'
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1,
    mode='max'
)

#### **Testing**

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score
)
model_builders = {
    'model_lstm': build_model_lstm,
    'model_two_way': build_model_1,
    'model_relu': build_model_2,
    'model_squareroot_log': build_model_3,
    'model_nonlin_taylor_newton_raphson_4': build_model_nonlin_optimized_4,
    'model_nonlin_taylor_newton_raphson_3': build_model_nonlin_optimized_3,
    'model_stabilized_log': build_model_7,
    'model_tanh': build_model_tanh,
    'model_crelu': build_model_crelu,
    'model_prelu': build_model_prelu,
    'model_elu': build_model_elu
}
detailed_results = []

for model_name, build_model_fn in model_builders.items():
    print(f"\nTraining {model_name}")

    model = build_model_fn()

    start_time = time.time()

    # Capture training history
    history = model.fit(X_train, y_train,
                       epochs=20,
                       batch_size=64,
                       validation_split=0.1,
                       callbacks=[lr_scheduler, early_stopping],
                       verbose=1)

    elapsed_time = time.time() - start_time

    # Evaluate on test set
    y_pred_probs = model.predict(X_test)
    y_pred_test = np.argmax(y_pred_probs, axis=1)
    y_true_test = np.argmax(y_test, axis=1)

    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_f1 = f1_score(y_true_test, y_pred_test, average='weighted')

    test_recall = recall_score(y_true_test, y_pred_test, average='weighted')
    test_precision = precision_score(y_true_test, y_pred_test, average='weighted')
    test_confusion = confusion_matrix(y_true_test, y_pred_test)
    test_auc = roc_auc_score(y_test, y_pred_probs, multi_class='ovr', average='weighted')
    test_log_loss = log_loss(y_test, y_pred_probs)
    start_inf = tf.timestamp()
    _ = model.predict(X_test, batch_size=32)
    end_inf = tf.timestamp()
    test_inference_time = float(end_inf - start_inf)

    # Store comprehensive results
    detailed_results.append({
        'Model': model_name,
        'Test_Accuracy': test_acc,
        'Test_F1_Score': test_f1,
        'Test_Recall_Score': test_recall,
        'Test_Precision_Score': test_precision,
        'Test_Confusion_Matrix': test_confusion,
        'Test_AUC_Score': test_auc,
        'Test_Log_Loss': test_log_loss,
        'Test_Inference_Time_sec': test_inference_time,

        'Train_Time_sec': elapsed_time,
        'Epochs_Trained': len(history.history['loss']),
        'Final_Train_Loss': history.history['loss'][-1],
        'Final_Val_Loss': history.history['val_loss'][-1],
        'Best_Val_Loss': min(history.history['val_loss']),
        'Train_Loss_History': history.history['loss'],
        'Val_Loss_History': history.history['val_loss'],
        'Overfitting_Gap': history.history['loss'][-1] - history.history['val_loss'][-1]
    })

In [ ]:
detailed_results = pd.DataFrame(detailed_results)
detailed_results

### **2way - cnn/lstm**

In [ ]:
def build_model_4():

    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)  # return_sequences so we can stack another LSTM

    # Second LSTM digs deeper into those patterns
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)  # stacking LSTMs helps with more complex time stuff
    x = tf.keras.layers.BatchNormalization()(lstm_out2)


    conv1 = tf.keras.layers.Conv1D(32, 3, padding='same')(x)
    pos1, neg1 = TwoLaneActivation()(conv1)
    x = tf.keras.layers.Concatenate()([pos1, neg1])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)  # <-- fixed

    conv2 = tf.keras.layers.Conv1D(64, 3, padding='same')(x)
    pos2, neg2 = TwoLaneActivation()(conv2)
    x = tf.keras.layers.Concatenate()([pos2, neg2])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)  # <-- fixed

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
build_model_4().summary()

#### **RELU - CNN/LSTM**

In [ ]:
def build_model_5():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    x = Conv1D(kernel_size=3, filters=32, padding="same", activation="relu")(lstm_out2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding="same")(x)

    x = Conv1D(kernel_size=3, filters=64, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding="same")(x)

    x = Flatten()(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
build_model_5().summary()

#### **Square root - Log activation -CNN/LSTM**

In [ ]:
def build_model_6():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    conv1= tf.keras.layers.Conv1D(32, 3, padding='same')(lstm_out2)
    x = Sqrtfunct()(conv1)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = Sqrtfunct()(conv2)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = Sqrtfunct()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Convolutional model with custom square root approximation using Newton-Raphson method and log approximation using 3 term taylor series - CNN/LSTM**

In [ ]:
def build_model_nonlin_optimized_lstm_cnn_3():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(lstm_out2)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
def build_model_nonlin_optimized_lstm_cnn_4():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(lstm_out2)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(12, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Stabilized Log - CNN/LSTM**

In [ ]:
def build_model_CNN_LSTM_stablized_log():
    class StabilizedLog(tf.keras.layers.Layer):
        def __init__(self, **kwargs):
            super(StabilizedLog, self).__init__(**kwargs)

        def call(self, inputs):
            abs_x = tf.abs(inputs)
            inner = inputs + abs_x + 1
            return tf.math.log(inner)

    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    conv1 = Conv1D(filters=64, kernel_size=3, padding='same')(lstm_out2)
    x = StabilizedLog()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = StabilizedLog()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = StabilizedLog()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model


### **LSTM AND Tanh Variation for CNN**

In [ ]:
def build_model_tanh_lstm_cnn():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)


    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = tf.math.tanh(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.math.tanh(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.math.tanh)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **CRELU Variation for CNN and LSTM**

In [ ]:
def build_model_crelu_lstm_cnn():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = tf.nn.crelu(conv1)  # concatenates ReLU(x) and ReLU(-x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.nn.crelu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = tf.nn.crelu(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **LSTM AND PRELU Variation for CNN**

In [ ]:
def build_model_prelu_lstm_cnn():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)


    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = PReLU()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = PReLU()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = PReLU()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **LSTM AND ELU Variation for CNN**

In [ ]:
def build_model_elu_lstm_cnn():
    inputs = tf.keras.Input(shape=(561,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)



    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = tf.keras.activations.elu(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.keras.activations.elu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.keras.activations.elu)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(12, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


#### **Training and Testing**

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import time

from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score
)

# --------------------------------------------------
# Models to test
# --------------------------------------------------
model_builders = {
    'model_two_way_linearity': build_model_1,
    'model_relu_linearity': build_model_2,
}

non_linearity = []

print("Evaluating All Models on the Test Set")

# --------------------------------------------------
# Callback: record FIRST and LAST epoch weights
# --------------------------------------------------
class WeightRecorder(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.first_epoch_weights = None
        self.last_epoch_weights = None

    def on_epoch_end(self, epoch, logs=None):
        weights = []

        for layer in self.model.layers:
            for w in layer.get_weights():
                weights.append(w.flatten())

        flat_weights = np.concatenate(weights) if weights else np.array([])

        if epoch == 0:
            self.first_epoch_weights = flat_weights.copy()

        self.last_epoch_weights = flat_weights.copy()

# --------------------------------------------------
# Training + Evaluation Loop
# --------------------------------------------------
for model_name, build_model_fn in model_builders.items():
    print(f"\nTraining {model_name}")

    model = build_model_fn()

    weight_recorder = WeightRecorder()

    start_time = time.time()
    history = model.fit(
        X_train,
        y_train,
        epochs=20,
        batch_size=64,
        validation_split=0.1,
        callbacks=[lr_scheduler, early_stopping, weight_recorder],
        verbose=1
    )
    elapsed_time = time.time() - start_time

    print(f"\nEvaluating {model_name}")

    y_pred_probs = model.predict(X_test)
    y_pred_test = np.argmax(y_pred_probs, axis=1)
    y_true_test = np.argmax(y_test, axis=1)

    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_prec = precision_score(y_true_test, y_pred_test, average='weighted')
    test_rec = recall_score(y_true_test, y_pred_test, average='weighted')
    test_f1 = f1_score(y_true_test, y_pred_test, average='weighted')
    test_confusion = confusion_matrix(y_true_test, y_pred_test)
    test_auc = roc_auc_score(y_test, y_pred_probs, multi_class='ovr', average='weighted')
    test_log_loss = log_loss(y_test, y_pred_probs)

    start_inf = tf.timestamp()
    _ = model.predict(X_test, batch_size=32)
    end_inf = tf.timestamp()
    test_inference_time = float(end_inf - start_inf)

    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test F1-score: {test_f1:.4f}")

    # --------------------------------------------------
    # Store results + weights
    # --------------------------------------------------
    non_linearity.append({
        'Model': model_name,
        'Test_Accuracy': test_acc,
        'Test_Precision': test_prec,
        'Test_Recall': test_rec,
        'Test_F1_Score': test_f1,
        'Test_AUC_Score': test_auc,
        'Test_Log_Loss': test_log_loss,
        'Test_Inference_Time_sec': test_inference_time,
        'Train_Time_sec': elapsed_time,
        'Epochs_Trained': len(history.history['loss']),
        'Final_Train_Loss': history.history['loss'][-1],
        'Final_Val_Loss': history.history['val_loss'][-1],
        'Best_Val_Loss': min(history.history['val_loss']),
        'Overfitting_Gap': history.history['loss'][-1] - history.history['val_loss'][-1],
        'First_Epoch_Weights': weight_recorder.first_epoch_weights,
        'Last_Epoch_Weights': weight_recorder.last_epoch_weights
    })

# --------------------------------------------------
# Convert to DataFrame
# --------------------------------------------------
non_linearity = pd.DataFrame(non_linearity)
non_linearity


In [ ]:
nonlinerity

In [ ]:
relu_row = non_linearity[non_linearity['Model'] == 'model_relu_linearity'].iloc[0]
two_row  = non_linearity[non_linearity['Model'] == 'model_two_way_linearity'].iloc[0]

W0_relu = relu_row['First_Epoch_Weights']
W1_relu = relu_row['Last_Epoch_Weights']

W0_two  = two_row['First_Epoch_Weights']
W1_two  = two_row['Last_Epoch_Weights']


In [ ]:
delta_relu = W1_relu - W0_relu
delta_two  = W1_two  - W0_two


In [ ]:
eps = 1e-8
ratio_relu = W1_relu / (W0_relu + eps)
ratio_two  = W1_two  / (W0_two  + eps)


In [ ]:
alpha_relu = np.dot(W0_relu, W1_relu) / np.dot(W0_relu, W0_relu)
alpha_two  = np.dot(W0_two,  W1_two)  / np.dot(W0_two,  W0_two)

alpha_relu, alpha_two


In [ ]:
comparison = pd.DataFrame({
    "Model": ["ReLU", "Two-Way"],
    "alpha": [alpha_relu, alpha_two],
    "p95_|ΔW|": [
        np.percentile(np.abs(delta_relu), 95),
        np.percentile(np.abs(delta_two), 95)
    ],
    "median_ratio": [
        np.median(ratio_relu),
        np.median(ratio_two)
    ]
})

comparison


In [ ]:
delta_W = delta_relu

frac_large_relu = np.mean(np.abs(delta_W) > 0.01)
frac_large_relu


In [ ]:
delta_W = delta_two

frac_large_two = np.mean(np.abs(delta_W) > 0.01)
frac_large_two


In [ ]:
model_builders = {
    #'model_2way_cnn_lstm': build_model_4,
    #'model_relu_cnn_lstm': build_model_5,
    #'model_cnn_lstm_sqaure_root': build_model_6,
    #'model_nonlin_optimized_lstm_cnn_3series': build_model_nonlin_optimized_lstm_cnn_3,
    #'model_nonlin_optimized_lstm_cnn_4series': build_model_nonlin_optimized_lstm_cnn_4,
    #'model_stab_log_lstm_cnn': build_model_CNN_LSTM_stablized_log,
    #'model_tanh_lstm_cnn': build_model_tanh_lstm_cnn,
    #'model_crelu_lstm_cnn': build_model_crelu_lstm_cnn,
    #'model_prelu_lstm_cnn': build_model_prelu_lstm_cnn,
    'model_elu_lstm_cnn': build_model_elu_lstm_cnn
}


#test_results_summary_hybrid = []

print("Evaluating All Models on the Test Set ")

for model_name, build_model_fn in model_builders.items():
    print(f"\nTraining {model_name}")


    model = build_model_fn()

    start_time = time.time()
    history = model.fit(X_train, y_train,
              epochs=20,
              batch_size=64,
              validation_split=0.1, #
              callbacks=[lr_scheduler, early_stopping],
              verbose=1)
    elapsed_time = time.time() - start_time

    print(f"\nEvaluating {model_name}")

    y_pred_probs = model.predict(X_test) #evaluate on X_test
    y_pred_test = np.argmax(y_pred_probs, axis=1)
    y_true_test = np.argmax(y_test, axis=1)

    #### TESTING

    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_prec = precision_score(y_true_test, y_pred_test, average='weighted')
    test_rec = recall_score(y_true_test, y_pred_test, average='weighted')
    test_f1 = f1_score(y_true_test, y_pred_test, average='weighted')
    test_confusion = confusion_matrix(y_true_test, y_pred_test)
    test_auc = roc_auc_score(y_test, y_pred_probs, multi_class='ovr', average='weighted')
    test_log_loss = log_loss(y_test, y_pred_probs)
    start_inf = tf.timestamp()
    _ = model.predict(X_test, batch_size=32)
    end_inf = tf.timestamp()
    test_inference_time = float(end_inf - start_inf)
    print(f"Test Accuracy for {model_name}: {test_acc:.4f}")
    print(f"Test F1-score for {model_name}: {test_f1:.4f}")

    test_results_summary_hybrid.append({
        'Model': model_name,
        'Test_Accuracy': test_acc,
        'Test_Precision': test_prec,
        'Test_Recall': test_rec,
        'confusion_matrix': test_confusion,
        'Test_AUC_Score': test_auc,
        'Test_Log_Loss': test_log_loss,
        'Test_Inference_Time_sec': test_inference_time,
        'Test_F1_Score': test_f1,
        'Train_Time_sec': elapsed_time,
        'Epochs_Trained': len(history.history['loss']),
        'Final_Train_Loss': history.history['loss'][-1],
        'Final_Val_Loss': history.history['val_loss'][-1],
        'Best_Val_Loss': min(history.history['val_loss']),
        'Train_Loss_History': history.history['loss'],
        'Val_Loss_History': history.history['val_loss'],
        'Overfitting_Gap': history.history['loss'][-1] - history.history['val_loss'][-1]
    })


In [ ]:
print("\nFinal Test Set Results")
detailed_results_2 = pd.DataFrame(test_results_summary_hybrid)
detailed_results_2

In [ ]:
detailed_results_2 = detailed_results_2.rename(columns={
    'Test_Recall': 'Test_Recall_Score',
    'Test_Precision': 'Test_Precision_Score',
    'confusion_matrix': 'Test_Confusion_Matrix'
})

detailed_results_2 = detailed_results_2[detailed_results.columns]


In [ ]:
final_dataframe_ucihapt = pd.concat([detailed_results, detailed_results_2], ignore_index=True)
final_dataframe_ucihapt['Model']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import rankdata

# ====================== PREP ======================
df = final_dataframe_ucihapt.copy()
df = df.reset_index(drop=True)

# Clean names
df['Model_Clean'] = (
    df['Model']
    .str.replace('model_', '', regex=False)
    .str.replace('_', ' ', regex=False)
    .str.title()
    .str.replace('Nonlin', 'Non-Linear')
    .str.replace('Crelu', 'C-ReLU')
    .str.replace('Prelu', 'PReLU')
    .str.replace('Elu', 'ELU')
    .str.replace('Lstm', 'LSTM')
    .str.replace('Two Way', 'Two-Way')
)

sns.set_style("whitegrid")

# =======================================================
# SHADE-A / SHADE-B LOGIC
# =======================================================

ordered_models = df['Model'].tolist()

shade_A_indices = [1, 3, 4, 5, 6, 11, 13, 14, 15, 16]

shade_A = "#1f77b4"
shade_B = "#A7D4FF"

colors = [
    shade_A if i in shade_A_indices else shade_B
    for i in range(len(ordered_models))
]

# ====================== BAR CHARTS — MAIN PERFORMANCE METRICS ======================
metrics_line = [
    ('Test_Accuracy',        'Test Accuracy'),
    ('Test_F1_Score',        'Test F1-Score'),
    ('Test_Precision_Score', 'Test Precision'),
    ('Test_Recall_Score',    'Test Recall'),
    ('Test_AUC_Score',       'Test AUC Score')
]

for col, title in metrics_line:
    plt.figure(figsize=(12, 7))

    plt.bar(
        df['Model_Clean'],
        df[col],
        color=colors,
        edgecolor='black',
        linewidth=1.4
    )

    plt.title(title, fontsize=20, pad=20)
    plt.ylabel(title)
    plt.xticks(rotation=45, ha='right')

    for x, y in zip(df['Model_Clean'], df[col]):
        plt.text(x, y + 0.002, f'{y:.4f}',
                 ha='center', fontsize=10, fontweight='bold')

    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# ====================== BAR CHARTS — COST / EFFICIENCY METRICS ======================
metrics_to_plot = [
    ('Test_Log_Loss',           'Test Log Loss (lower better)'),
    ('Test_Inference_Time_sec', 'Inference Time (seconds)'),
    ('Train_Time_sec',          'Training Time (seconds)'),
    ('Best_Val_Loss',           'Best Validation Loss'),
    ('Overfitting_Gap',         'Overfitting Gap (negative = good)'),
]

for col, title in metrics_to_plot:
    plt.figure(figsize=(11, 7))

    bars = plt.bar(
        df['Model_Clean'],
        df[col],
        color=colors,
        edgecolor='black',
        linewidth=1.4
    )

    plt.title(title, fontsize=20, pad=20)
    plt.ylabel(title.split('(')[0].strip())
    plt.xticks(rotation=45, ha='right')

    for bar, val in zip(bars, df[col]):
        h = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            h + abs(h)*0.02,
            f'{val:.4f}' if col != 'Test_Inference_Time_sec' else f'{val:.3f}s',
            ha='center',
            va='bottom',
            fontweight='bold'
        )

    plt.tight_layout()
    plt.show()


In [ ]:
final_dataframe_ucihapt

#### Load cached results (optional)

Use this section only when loading previously saved CSV outputs instead of re-running training.


In [ ]:

final_dataframe_ucihapt = pd.read_csv('ucihapt.csv')

In [ ]:
final_dataframe_wisdm = pd.read_csv('wisdm.csv')

In [ ]:
import pandas as pd
import numpy as np
import re

# Load your broken CSV
df = pd.read_csv("ucihapt.csv")  # <<<--- CHANGE TO YOUR ACTUAL FILENAME

print(f"Loaded {len(df)} rows. Starting recovery debug...\n")

def recover_confusion_matrix_from_row(row):
    # Step 1: Concatenate all values in the row into one string (they're smashed)
    full_string = ' '.join([str(val) for val in row if pd.notna(val)])

    # Debug: print the first row's full string so we can see what's there
    if row.name == 0:
        print("=== DEBUG: Full smashed string from row 0 ===")
        print(full_string[:1000] + "..." if len(full_string) > 1000 else full_string)
        print("=== END DEBUG ===\n")

    # Step 2: Find the [[ ... ]] block (even if truncated)
    match = re.search(r'\[\[(.*?)\]\]', full_string, re.DOTALL)
    if not match:
        return None

    inner = match.group(1)

    # Remove ... if present
    inner = inner.replace('...', '')

    # Split into rows using common patterns: "] [" or newlines or just ]
    rows_str = re.split(r'\]\s*\[|\n|\]', inner)

    rows = []
    for r in rows_str:
        r = re.sub(r'[\[\]]', '', r).strip()
        if not r:
            continue
        # Split on any whitespace
        values = [int(x) for x in re.split(r'\s+', r) if x.isdigit() or (x.startswith('-') and x[1:].isdigit())]
        if len(values) > 0:
            rows.append(values)

    # Keep only rows with exactly 12 values (strict to avoid garbage)
    valid_rows = [r for r in rows if len(r) == 12]

    if len(valid_rows) >= 6:  # at least half the matrix to be useful
        # Pad with zeros if some rows are missing (optional)
        while len(valid_rows) < 12:
            valid_rows.append([0] * 12)
        return np.array(valid_rows[:12], dtype=int)

    return None

# Create new column with recovered matrices
df['Recovered_Confusion_Matrix'] = df.apply(recover_confusion_matrix_from_row, axis=1)

# Replace the old column if recovery worked
df['Test_Confusion_Matrix'] = df['Recovered_Confusion_Matrix']

# Report
recovered_count = df['Test_Confusion_Matrix'].notna().sum()
print(f"\nRecovery complete!")
print(f"Successfully recovered confusion matrices for {recovered_count}/21 rows")

if recovered_count > 0:
    example = df['Test_Confusion_Matrix'].dropna().iloc[0]
    print(f"Example recovered shape: {example.shape}")
    print("Example diagonal (true positives):", np.diag(example))
    print("\nFirst few rows of recovered matrix:")
    print(example[:5])  # show top 5 rows

# Save the improved version
df.to_csv("final_dataframe_ucihapt.csv", index=False)
df.to_pickle("final_dataframe_ucihapt.pkl")
print("\nSaved as final_dataframe_ucihapt.csv and .pkl")

final_dataframe_ucihapt = df
print("\nNow use: final_dataframe_ucihapt")

In [ ]:
import pandas as pd
import numpy as np
import re

# Load the WISDM broken CSV
df = pd.read_csv("wisdm.csv")  # Make sure the file is named exactly "wisdm.csv"

print(f"Loaded {len(df)} rows from wisdm.csv. Starting recovery debug...\n")

def recover_confusion_matrix_from_row(row):
    # Step 1: Concatenate all values in the row into one big string
    full_string = ' '.join([str(val) for val in row if pd.notna(val)])

    # Debug: Print the full smashed string from the first row
    if row.name == 0:
        print("=== DEBUG: Full smashed string from row 0 (WISDM) ===")
        print(full_string[:1000] + ("..." if len(full_string) > 1000 else ""))
        print("=== END DEBUG ===\n")

    # Step 2: Find the [[ ... ]] block using regex (handles truncation)
    match = re.search(r'\[\[(.*?)\]\]', full_string, re.DOTALL)
    if not match:
        return None

    inner = match.group(1)

    # Remove truncation markers
    inner = inner.replace('...', '')

    # Split rows using common NumPy 2D patterns
    rows_str = re.split(r'\]\s*\[|\n|\]', inner)

    rows = []
    for r in rows_str:
        r = re.sub(r'[\[\]]', '', r).strip()
        if not r:
            continue
        # Split on whitespace, convert to int (handles negative numbers too)
        values = [int(x) for x in re.split(r'\s+', r)
                  if x.lstrip('-').isdigit()]
        if len(values) > 0:
            rows.append(values)

    # Keep only rows with exactly the expected number of classes
    # For WISDM dataset, there are typically 18 activities (adjust if different)
    # Common WISDM activity count is 18 or 6 depending on version
    # We'll be flexible: accept rows with 6, 18, or 12 values
    possible_sizes = [6, 12, 18]
    valid_rows = [r for r in rows if len(r) in possible_sizes]

    if len(valid_rows) >= 4:  # at least a few good rows
        # Determine the class count from the first valid row
        n_classes = len(valid_rows[0])
        # Keep only rows matching that size
        valid_rows = [r for r in valid_rows if len(r) == n_classes]
        # Pad to full size with zeros if needed
        while len(valid_rows) < n_classes:
            valid_rows.append([0] * n_classes)
        return np.array(valid_rows[:n_classes], dtype=int)

    return None

# Recover confusion matrices
df['Recovered_Confusion_Matrix'] = df.apply(recover_confusion_matrix_from_row, axis=1)

# Replace the original column
df['Test_Confusion_Matrix'] = df['Recovered_Confusion_Matrix']

# Report results
recovered_count = df['Test_Confusion_Matrix'].notna().sum()
print(f"\nRecovery complete for WISDM!")
print(f"Successfully recovered confusion matrices for {recovered_count}/{len(df)} rows")

if recovered_count > 0:
    example = df['Test_Confusion_Matrix'].dropna().iloc[0]
    print(f"Example recovered shape: {example.shape}")
    print("Example diagonal (true positives):", np.diag(example))
    print("\nFirst few rows of recovered matrix:")
    print(example[:min(8, example.shape[0])])  # show up to 8 rows

# Save the fixed versions
df.to_csv("final_dataframe_wisdm.csv", index=False)
df.to_pickle("final_dataframe_wisdm.pkl")
print("\nSaved as final_dataframe_wisdm.csv and final_dataframe_wisdm.pkl")

# Assign to variable
final_dataframe_wisdm = df
print("\nfinal_dataframe_wisdm")

In [ ]:

def confusion_to_class_metrics(cm):
    cm = np.array(cm)

    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    eps = 1e-9

    precision = TP / (TP + FP + eps)
    recall    = TP / (TP + FN + eps)
    f1        = 2 * (precision * recall) / (precision + recall + eps)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


def build_classwise_comparison(df, class_names, metric="f1"):
    comparison = {cls: {} for cls in class_names}

    for _, row in df.iterrows():
        model_name = row["Model"]
        cm = row["Test_Confusion_Matrix"]

        # Skip invalid / missing confusion matrices
        if cm is None:
            continue

        metrics = confusion_to_class_metrics(cm)

        for i, cls in enumerate(class_names):
            comparison[cls][model_name] = metrics[metric][i]

    return pd.DataFrame(comparison).T


def best_model_per_activity(comparison_df):
    return comparison_df.idxmax(axis=1)


# ----------------------------------
# Class labels
# ----------------------------------

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
    "STAND_TO_SIT",
    "SIT_TO_STAND",
    "SIT_TO_LIE",
    "LIE_TO_SIT",
    "STAND_TO_LIE",
    "LIE_TO_STAND"
]


comparison_df = build_classwise_comparison(
    final_dataframe_ucihapt,
    class_names,
    metric="f1"
)


best_models = best_model_per_activity(comparison_df).reset_index()
best_models.columns = ["Activity", "Best_Model"]


In [ ]:
best_models

In [ ]:
cm = final_dataframe_ucihapt["Test_Confusion_Matrix"].dropna().iloc[0]
cm = np.array(cm)

# Row-wise sum gives number of true samples per class
class_samples = pd.Series(
    cm.sum(axis=1),
    index=comparison_df.index
)

# Optional: percentage support
class_support = class_samples / class_samples.sum() * 100

# Combine into one table
class_distribution = pd.DataFrame({
    "Samples": class_samples,
    "Support (%)": class_support
})

print("\n==============================")
print(" TEST SET CLASS DISTRIBUTION ")
print("==============================\n")
print(class_distribution)


In [ ]:
comparison_analysis = best_models.set_index("Activity").join(class_distribution)

comparison_analysis = comparison_analysis.sort_values("Support (%)", ascending=False)
comparison_analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Copy dataframe
df = comparison_analysis.copy()

# Proposed models list
proposed_models = [
    "model_two_way",
    "model_squareroot_log",
    "model_nonlin_taylor_newton_raphson_4",
    "model_nonlin_taylor_newton_raphson_3",
    "model_stabilized_log",
    "model_2way_cnn_lstm",
    "model_cnn_lstm_sqaure_root",
    "model_nonlin_optimized_lstm_cnn_3series",
    "model_nonlin_optimized_lstm_cnn_4series",
    "model_stab_log_lstm_cnn"
]

# Assign colors
shade_A = "#1f77b4"  # deep blue
shade_B = "#A7D4FF"  # light blue
df["Color"] = df["Best_Model"].apply(lambda x: shade_A if x in proposed_models else shade_B)

# Sort by Samples for horizontal bars
df_sorted = df.sort_values("Samples", ascending=True)

# Plot
plt.figure(figsize=(12, 8))
bars = plt.barh(df_sorted.index, df_sorted["Samples"], color=df_sorted["Color"])

# Annotate inside or just outside the bar
for bar, (_, row) in zip(bars, df_sorted.iterrows()):
    width = bar.get_width()
    label_x_pos = width - 20 if width > 60 else width + 5  # inside if wide enough, else outside
    ha = 'right' if width > 60 else 'left'
    plt.text(label_x_pos, bar.get_y() + bar.get_height()/2,
             f"{row['Best_Model']} ({row['Support (%)']:.1f}%)",
             va='center', ha=ha, fontsize=10, fontweight='bold')

# Labels and title
plt.xlabel("Number of Samples", fontsize=12, fontweight='bold')
plt.ylabel("Activity", fontsize=12, fontweight='bold')
plt.title("Class-wise Best Model (Proposed vs Other Models)", fontsize=14, fontweight='bold')

# Legend
legend_elements = [
    mpatches.Patch(color=shade_A, label="Proposed Models"),
    mpatches.Patch(color=shade_B, label="Other Models")
]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()


# **Unanimous Improvement Ratio (UIR)**

## Definition
UIR is a **weight-independent method** to compare the performance of multiple models across several evaluation metrics. Instead of aggregating metrics with arbitrary weights, UIR counts the proportion of test cases where one model outperforms another on **all metrics simultaneously**.

Formally, for two models \(A\) and \(B\):

\[
\text{UIR}(A, B) = \frac{\text{Number of test cases where } A \ge B \text{ on all metrics}}{\text{Total number of test cases}}
\]

- If \(A\) is better than or equal to \(B\) in **every metric** for a test case, it contributes to the numerator.
- The resulting ratio ranges from 0 to 1:
  - **1** → Model \(A\) unanimously outperforms \(B\) on all metrics.
  - **0** → Model \(A\) never outperforms \(B\) on all metrics.

## Key Advantages
- **Weight-independent:** Avoids assigning subjective weights to metrics like accuracy, F1-score, or AUC.
- **Robust across metrics:** Highlights models that consistently perform well across all evaluation measures.
- **Interpretable ranking:** By computing UIR for all pairs of models, models can be ranked based on how often they dominate others.

## Application to Your Study
- Each of the 21 models is compared **pairwise** using multiple metrics (e.g., F1-score, AUC, precision, recall).
- UIR counts how many times a model **dominates all other models** across all metrics.
- Models with the highest UIR are considered **consistently superior**.

## Citation
Amigó, E., Gonzalo, J., Artiles, J., & Verdejo, F. (2011). *Combining evaluation metrics via the unanimous improvement ratio and its application to clustering tasks.* Journal of Artificial Intelligence Research, 42, 689–718.


In [ ]:
from itertools import combinations
from sklearn.preprocessing import MinMaxScaler

# -------------------------------
# Define metrics
# -------------------------------
perf_metrics = ["Test_Accuracy", "Test_F1_Score", "Test_Recall_Score", "Test_Precision_Score", "Test_AUC_Score"]
cost_metrics = ["Train_Time_sec", "Test_Inference_Time_sec", "Test_Log_Loss", "Overfitting_Gap"]

df = final_dataframe_ucihapt.set_index("Model").copy()

# -------------------------------
# Normalize metrics
# -------------------------------
scaler = MinMaxScaler()

# Performance metrics: higher is better
df_perf_norm = pd.DataFrame(
    scaler.fit_transform(df[perf_metrics]),
    columns=perf_metrics,
    index=df.index
)

# Cost metrics: lower is better, so invert after normalization
df_cost_norm = pd.DataFrame(
    1 - scaler.fit_transform(df[cost_metrics]),
    columns=cost_metrics,
    index=df.index
)

# -------------------------------
# Function to compute UIR
# -------------------------------
def compute_uir(metrics_df):
    model_names = metrics_df.index.tolist()
    uir_matrix = pd.DataFrame(
        np.zeros((len(model_names), len(model_names))),
        index=model_names,
        columns=model_names
    )

    for model_i, model_j in combinations(model_names, 2):
        metrics_i = metrics_df.loc[model_i].values
        metrics_j = metrics_df.loc[model_j].values

        unanimous = metrics_i >= metrics_j
        if unanimous.all():
            uir_matrix.loc[model_i, model_j] = 1
        elif (metrics_j >= metrics_i).all():
            uir_matrix.loc[model_j, model_i] = 1
        else:
            pass  # neither dominates

    uir_summary = uir_matrix.sum(axis=1).sort_values(ascending=False)
    uir_ranking = uir_summary.rank(ascending=False, method='min')

    return uir_summary, uir_ranking

# -------------------------------
# Compute UIR for performance and cost separately
# -------------------------------
perf_uir_summary, perf_uir_ranking = compute_uir(df_perf_norm)
cost_uir_summary, cost_uir_ranking = compute_uir(df_cost_norm)

# -------------------------------
# Display results
# -------------------------------
print("\n==============================")
print("PERFORMANCE UIR SUMMARY")
print("==============================\n")
print(perf_uir_summary)
print("\nPerformance UIR ranking:\n", perf_uir_ranking)

print("\n==============================")
print("COST/EFFICIENCY UIR SUMMARY")
print("==============================\n")
print(cost_uir_summary)
print("\nCost/Efficiency UIR ranking:\n", cost_uir_ranking)

In [ ]:
df_plot = pd.DataFrame({
    "Performance_UIR": perf_uir_summary,
    "Cost_UIR": cost_uir_summary
})

# Sort for better visualization
df_perf_sorted = df_plot.sort_values("Performance_UIR", ascending=False)
df_cost_sorted = df_plot.sort_values("Cost_UIR", ascending=False)

# Define your proposed models
proposed_models = [
    "model_two_way", "model_squareroot_log", "model_nonlin_taylor_newton_raphson_4",
    "model_nonlin_taylor_newton_raphson_3", "model_stabilized_log", "model_2way_cnn_lstm",
    "model_cnn_lstm_sqaure_root", "model_nonlin_optimized_lstm_cnn_3series",
    "model_nonlin_optimized_lstm_cnn_4series", "model_stab_log_lstm_cnn"
]

# Assign colors based on model type
shade_A = "#1f77b4"  # proposed models
shade_B = "#A7D4FF"  # existing models

df_perf_sorted["Color"] = df_perf_sorted.index.to_series().apply(lambda x: shade_A if x in proposed_models else shade_B)
df_cost_sorted["Color"] = df_cost_sorted.index.to_series().apply(lambda x: shade_A if x in proposed_models else shade_B)

# -------------------------------
# Performance UIR - vertical bar chart
# -------------------------------
plt.figure(figsize=(18, 8))
plt.bar(df_perf_sorted.index, df_perf_sorted["Performance_UIR"], color=df_perf_sorted["Color"], edgecolor='black')
plt.xticks(rotation=90, fontsize=10)
plt.ylabel("Performance UIR", fontsize=12, fontweight='bold')
plt.title("Performance UIR per Model", fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------------
# Cost/Efficiency UIR - vertical bar chart
# -------------------------------
plt.figure(figsize=(18, 8))
plt.bar(df_cost_sorted.index, df_cost_sorted["Cost_UIR"], color=df_cost_sorted["Color"], edgecolor='black')
plt.xticks(rotation=90, fontsize=10)
plt.ylabel("Cost/Efficiency UIR", fontsize=12, fontweight='bold')
plt.title("Cost/Efficiency UIR per Model", fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


# UIR Analysis: Performance vs Cost/Efficiency

We computed **Unanimous Improvement Ratio (UIR)** for the 21 models using two separate sets of metrics:

- **Performance metrics:** Accuracy, F1-score, Recall, Precision, AUC
- **Cost/Efficiency metrics:** Training time, Inference time, Log Loss, Overfitting gap

This allows us to assess **predictive quality** and **resource efficiency** independently.

---

## 1. Performance UIR

**Top performers:**

- `model_two_way` dominates most other models with a UIR of 20.
- `model_stabilized_log` and `model_elu` are also strong performers.

**Interpretation:**
Models with higher performance UIR **consistently outperform others** across all predictive metrics.
For example, `model_two_way` shows robust predictive accuracy, high F1-score, and AUC across all classes.

**Observation:**
Some hybrid or LSTM-CNN models (`model_tanh_lstm_cnn`, `model_nonlin_optimized_lstm_cnn_3series`) rank lowest in performance UIR, indicating **inconsistent predictive performance** across metrics.

---

## 2. Cost/Efficiency UIR

**Top efficient models:**

- `model_two_way`, `model_relu`, `model_stabilized_log`, and `model_prelu` dominate in cost/efficiency, achieving high UIR scores.

**Interpretation:**
These models not only perform well but also **train faster, infer faster, and have lower log loss and overfitting gap**.

**Observation:**
Complex architectures such as LSTM-CNN hybrids or models with Taylor-Newton nonlinearities rank low, reflecting their **higher computational cost or overfitting tendencies**.

---

## 3. Combined Insight

- `model_two_way` stands out in **both performance and cost-efficiency**, making it **consistently superior**.
- Some models like `model_elu` or `model_prelu` are **strong in predictive performance** but slightly lower in cost-efficiency.
- Models with low UIR in both dimensions (e.g., `model_tanh_lstm_cnn`, `model_nonlin_optimized_lstm_cnn_3series`) are **less desirable for practical deployment**.


# **PARETO MULTI OBJECTIVE OPTIMAL TEST**

In [ ]:
# -------------------------------
# Compute aggregate performance and cost scores for Pareto analysis
# -------------------------------
df_combined = pd.DataFrame({
    "Performance_Score": df_perf_norm.mean(axis=1),   # average normalized performance
    "Cost_Score": df_cost_norm.mean(axis=1)           # average normalized efficiency (higher is better after inversion)
})

# -------------------------------
# Function to compute Pareto front
# -------------------------------
def pareto_front(df):
    is_dominated = np.zeros(len(df), dtype=bool)

    for i, row_i in df.iterrows():
        for j, row_j in df.iterrows():
            if i == j:
                continue
            # Check if j dominates i
            if (row_j["Performance_Score"] >= row_i["Performance_Score"]) and \
               (row_j["Cost_Score"] >= row_i["Cost_Score"]) and \
               ((row_j["Performance_Score"] > row_i["Performance_Score"]) or (row_j["Cost_Score"] > row_i["Cost_Score"])):
                is_dominated[df_combined.index.get_loc(i)] = True
                break
    return df[~is_dominated]

pareto_models = pareto_front(df_combined)
print("\nPareto-optimal models:\n", pareto_models)


In [ ]:
pareto_df = pd.DataFrame({
    "Performance_Score": df_perf_norm.mean(axis=1),  # or your normalized performance score
    "Cost_Score": df_cost_norm.mean(axis=1)          # or your normalized cost score
})

# Ideal point (both performance and cost/efficiency maxed)
ideal_point = np.array([1, 1])

# Compute Euclidean distance to ideal point for each model
pareto_df["Distance_to_Ideal"] = np.sqrt(
    (pareto_df["Performance_Score"] - ideal_point[0])**2 +
    (pareto_df["Cost_Score"] - ideal_point[1])**2
)

# Rank models: smaller distance = better
pareto_df["Rank"] = pareto_df["Distance_to_Ideal"].rank(method="min")

# Sort by rank
pareto_df_sorted = pareto_df.sort_values("Rank")

print("\nModel ranking based on distance to Pareto-optimal point:\n")
print(pareto_df_sorted)

In [ ]:
import matplotlib.pyplot as plt
from adjustText import adjust_text  # pip install adjustText

# -------------------------------
# Prepare data
# -------------------------------
df_plot = pareto_df_sorted.copy()

pareto_mask = df_plot.index.isin(pareto_models.index)
top5_mask = df_plot.head(5).index

# Highlight only: Pareto-optimal + Top 5 closest (union)
highlight_models = df_plot.loc[pareto_mask | df_plot.index.isin(top5_mask)]

# -------------------------------
# Plot
# -------------------------------
plt.figure(figsize=(12, 9))
plt.style.use('default')  # Clean look

# 1. Pareto-optimal models
pareto_points = highlight_models[highlight_models.index.isin(pareto_models.index)]
plt.scatter(pareto_points["Cost_Score"], pareto_points["Performance_Score"],
            c='#d62728', s=180, edgecolors='black', linewidth=1.2,
            label='Pareto-optimal', zorder=10)

# 2. Top-5 closest (only if not already Pareto)
top5_only = highlight_models[highlight_models.index.isin(top5_mask)]
top5_only = top5_only[~top5_only.index.isin(pareto_models.index)]

plt.scatter(top5_only["Cost_Score"], top5_only["Performance_Score"],
            c='#1f77b4', s=160, edgecolors='black', linewidth=1,
            label='Top-5 closest to ideal', zorder=9)

# 3. Smart labels
texts = []
for model_name, row in highlight_models.iterrows():
    color = '#d62728' if model_name in pareto_models.index else '#1f77b4'
    weight = 'bold' if model_name in pareto_models.index else 'normal'
    texts.append(
        plt.text(row["Cost_Score"], row["Performance_Score"], f" {model_name}",
                 fontsize=10.5, fontweight=weight, color=color,
                 ha='left', va='center')
    )

adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.8, alpha=0.6))

# 4. Ideal point
plt.scatter(1.0, 1.0, marker=(5, 1, 0), s=300, c='gold', edgecolors='black', linewidth=1.5,
            label='Ideal point (1.0, 1.0)', zorder=11)

# -------------------------------
# Styling and zoom axes to top models only
# -------------------------------
margin = 0.02
plt.xlim(highlight_models["Cost_Score"].min() - margin, highlight_models["Cost_Score"].max() + margin)
plt.ylim(highlight_models["Performance_Score"].min() - margin, highlight_models["Performance_Score"].max() + margin)

plt.xlabel("Cost/Efficiency Score (Higher → Better)", fontsize=13, fontweight='bold')
plt.ylabel("Performance Score (Higher → Better)", fontsize=13, fontweight='bold')
plt.title("Pareto-optimal & Top-3 Closest Models (Zoomed View)",
          fontsize=16, fontweight='bold', pad=20)

plt.grid(True, alpha=0.3)
plt.legend(loc='lower left', frameon=True, fancybox=True, shadow=True)
plt.tight_layout()
plt.show()


## Pareto Analysis

Pareto analysis is a technique used in multi-objective optimization to identify solutions that are **non-dominated**, meaning no other solution is better in all objectives simultaneously. In the context of model evaluation, a Pareto-optimal model is one for which **no other model has both higher performance and higher efficiency**.

This allows decision-makers to select models that provide the best trade-offs between competing objectives such as predictive accuracy and computational cost.

**Citation:**
Kasprzak, E. M., & Lewis, K. E. (2001). *Pareto analysis in multiobjective optimization using the collinearity theorem and scaling method.* Structural and Multidisciplinary Optimization, 22(3), 208–218.


In [ ]:
ucihapt = final_dataframe_ucihapt
ucihapt.to_csv("ucihapt.csv", index=False)

# **WISDM**

In [ ]:
lines = []

with open('wisdm.arff', 'r') as f:
    start_data = False
    for line in f:
        if line.strip().lower() == '@data':
            start_data = True
            continue
        if start_data:
            lines.append(line.strip())
data_wisdm = pd.DataFrame([row.split(',') for row in lines])

In [ ]:
train_wisdm, temp_wisdm = train_test_split(data_wisdm, test_size=0.3, random_state=42)
val_wisdm, test_wisdm = train_test_split(temp_wisdm, test_size=0.5, random_state=42)

In [ ]:
train_wisdm.isnull().sum()

In [ ]:
train_wisdm.columns

In [ ]:
train_wisdm

In [ ]:
y_train_wisdm = train_wisdm.iloc[:, 45]
X_train_wisdm = train_wisdm.drop([0,45], axis=1)

y_val_wisdm = val_wisdm.iloc[:, 45]
X_val_wisdm= val_wisdm.drop([0,45], axis=1)

y_test_wisdm = test_wisdm.iloc[:, 45]
X_test_wisdm = test_wisdm.drop([0,45], axis=1)

In [ ]:
X_train_wisdm = X_train_wisdm.apply(pd.to_numeric, errors='coerce')
X_test_wisdm = X_test_wisdm.apply(pd.to_numeric, errors='coerce')
X_val_wisdm = X_val_wisdm.apply(pd.to_numeric, errors='coerce')

In [ ]:
X_train_wisdm = X_train_wisdm.fillna(X_train_wisdm.mean())
X_val_wisdm = X_val_wisdm.fillna(X_train_wisdm.mean())
X_test_wisdm = X_test_wisdm.fillna(X_train_wisdm.mean())

In [ ]:
X_train_wisdm.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
# Scaling
scaler = StandardScaler()
X_train_wisdm = scaler.fit_transform(X_train_wisdm)
X_test_wisdm  = scaler.transform(X_test_wisdm)
X_val_wisdm   = scaler.transform(X_val_wisdm)

# Reshape to (samples, 44, 1)
X_train_wisdm = X_train_wisdm.reshape(-1, 44, 1)
X_test_wisdm  = X_test_wisdm.reshape(-1, 44, 1)
X_val_wisdm   = X_val_wisdm.reshape(-1, 44, 1)

# Label encoding
from sklearn.preprocessing import LabelEncoder
encoding_wisdm = LabelEncoder()
y_train_wisdm = encoding_wisdm.fit_transform(y_train_wisdm)
y_val_wisdm   = encoding_wisdm.transform(y_val_wisdm)
y_test_wisdm  = encoding_wisdm.transform(y_test_wisdm)

# One-hot encode labels (required for categorical_crossentropy)
from tensorflow.keras.utils import to_categorical
y_train_wisdm = to_categorical(y_train_wisdm, num_classes=6)
y_val_wisdm   = to_categorical(y_val_wisdm,   num_classes=6)
y_test_wisdm  = to_categorical(y_test_wisdm,  num_classes=6)

#### **TWO WAY**


In [ ]:
def build_model_1_wisdm():
    inputs = tf.keras.Input(shape=(44,1))

    conv1 = tf.keras.layers.Conv1D(32, 3, padding='same')(inputs)
    pos1, neg1 = TwoLaneActivation()(conv1)
    x = tf.keras.layers.Concatenate()([pos1, neg1])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = tf.keras.layers.Conv1D(64, 3, padding='same')(x)
    pos2, neg2 = TwoLaneActivation()(conv2)
    x = tf.keras.layers.Concatenate()([pos2, neg2])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


#### **RELU - CNN**


In [ ]:
def build_model_2_wisdm():
    model = Sequential([
        Conv1D(kernel_size=3, filters=32, padding="same", activation="relu", input_shape=(44,1)),
        BatchNormalization(),
        MaxPooling1D(pool_size=2, padding="same"),

        Conv1D(kernel_size=3, filters=64, padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2, padding="same"),

        Flatten(),
        Dense(512, activation="relu"),
        Dropout(0.5),
        Dense(6, activation="softmax")
    ])

    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


#### **Square root - Log activation - CNN**


In [ ]:
def build_model_3_wisdm():
    inputs = tf.keras.Input(shape=(44,1))

    conv1 = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = Sqrtfunct()(conv1)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = Sqrtfunct()(conv2)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = Sqrtfunct()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Nonlinear Optimized 4**


In [ ]:
from tensorflow.keras.optimizers import legacy
def build_model_nonlin_optimized_4_wisdm():
    inputs = tf.keras.Input(shape=(44,1))


    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Nonlinear Optimized 3**


In [ ]:
from tensorflow.keras.optimizers import legacy
def build_model_nonlin_optimized_3_wisdm():
    inputs = tf.keras.Input(shape=(44,1))


    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Stabilized Log - CNN**


In [ ]:
def build_model_7_wisdm():

    inputs = Input(shape=(44,1))

    conv1 = Conv1D(filters=32, kernel_size=3, padding='same')(inputs)
    x = StabilizedLog()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = StabilizedLog()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = StabilizedLog()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model


### **Tanh Variation**


In [ ]:
def build_model_tanh_wisdm():
    inputs = Input(shape=(44,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = tf.math.tanh(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.math.tanh(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.math.tanh)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **CRELU Variation**


In [ ]:
def build_model_crelu_wisdm():
    inputs = Input(shape=(44,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = tf.nn.crelu(conv1)  # concatenates ReLU(x) and ReLU(-x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.nn.crelu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = tf.nn.crelu(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **PRELU Variation**


In [ ]:
from tensorflow.keras.layers import PReLU

def build_model_prelu_wisdm():
    inputs = Input(shape=(44,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = PReLU()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = PReLU()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = PReLU()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **ELU Variation**


In [ ]:
def build_model_elu_wisdm():
    inputs = Input(shape=(44,1))

    conv1 = Conv1D(32, 3, padding='same')(inputs)
    x = tf.keras.activations.elu(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.keras.activations.elu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.keras.activations.elu)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


#### **LSTM**


In [ ]:
from tensorflow.keras.layers import Attention
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_model_lstm_wisdm():
    inputs = Input(shape=(44,1))

    # LSTM with sequences
    x = LSTM(128, return_sequences=True)(inputs)
    x = BatchNormalization()(x)

    x = LSTM(128, return_sequences=True)(x)
    x = BatchNormalization()(x)

    # Attention: query and value = same sequence (self-attention)
    attention_output = Attention()([x, x])

    # Pooling after attention
    x = tf.reduce_mean(attention_output, axis=1)

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)

    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model


### **2way - cnn/lstm**


In [ ]:
def build_model_4_wisdm():

    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)  # return_sequences so we can stack another LSTM

    # Second LSTM digs deeper into those patterns
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)  # stacking LSTMs helps with more complex time stuff
    x = tf.keras.layers.BatchNormalization()(lstm_out2)


    conv1 = tf.keras.layers.Conv1D(32, 3, padding='same')(x)
    pos1, neg1 = TwoLaneActivation()(conv1)
    x = tf.keras.layers.Concatenate()([pos1, neg1])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)  # <-- fixed

    conv2 = tf.keras.layers.Conv1D(64, 3, padding='same')(x)
    pos2, neg2 = TwoLaneActivation()(conv2)
    x = tf.keras.layers.Concatenate()([pos2, neg2])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)  # <-- fixed

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


#### **RELU - CNN/LSTM**


In [ ]:
def build_model_5_wisdm():
    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    x = Conv1D(kernel_size=3, filters=32, padding="same", activation="relu")(lstm_out2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding="same")(x)

    x = Conv1D(kernel_size=3, filters=64, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding="same")(x)

    x = Flatten()(x)
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

#### **Square root - Log activation -CNN/LSTM**


In [ ]:
def build_model_6_wisdm():
    class Sqrtfunct(tf.keras.layers.Layer):
        def __init__(self, **kwargs):
            super(Sqrtfunct, self).__init__(**kwargs)
        def call(self, inputs):
             abs_x = tf.abs(inputs)
             transformed = inputs + abs_x
             transformed = tf.add(transformed, tf.math.log1p(abs_x))
             return tf.where(inputs > 0, tf.sqrt(transformed), transformed)

    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    conv1= tf.keras.layers.Conv1D(32, 3, padding='same')(lstm_out2)
    x = Sqrtfunct()(conv1)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = Sqrtfunct()(conv2)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = Sqrtfunct()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Convolutional model with custom square root**


In [ ]:
def build_model_nonlin_optimized_lstm_cnn_3_wisdm():
    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(lstm_out2)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_3()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Convolutional model with custom square root**


In [ ]:
def build_model_nonlin_optimized_lstm_cnn_4_wisdm():
    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    x = tf.keras.layers.Conv1D(filters=32, kernel_size=3, padding='same')(lstm_out2)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2, padding='same')(x)


    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128)(x)
    x = SqrtfunctNonlinear_4()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=legacy.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


#### **Stabilized Log - CNN/LSTM**


In [ ]:
def build_model_CNN_LSTM_stablized_log_wisdm():
    class StabilizedLog(tf.keras.layers.Layer):
        def __init__(self, **kwargs):
            super(StabilizedLog, self).__init__(**kwargs)

        def call(self, inputs):
            abs_x = tf.abs(inputs)
            inner = inputs + abs_x + 1
            return tf.math.log(inner)

    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    conv1 = Conv1D(filters=64, kernel_size=3, padding='same')(lstm_out2)
    x = StabilizedLog()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(filters=64, kernel_size=3, padding='same')(x)
    x = StabilizedLog()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = StabilizedLog()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model


### **LSTM AND Tanh Variation for CNN**


In [ ]:
def build_model_tanh_lstm_cnn_wisdm():
    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)


    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = tf.math.tanh(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.math.tanh(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.math.tanh)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **CRELU Variation for CNN and LSTM**


In [ ]:
def build_model_crelu_lstm_cnn_wisdm():
    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)

    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = tf.nn.crelu(conv1)  # concatenates ReLU(x) and ReLU(-x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.nn.crelu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = tf.nn.crelu(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **LSTM AND PRELU Variation for CNN**


In [ ]:
def build_model_prelu_lstm_cnn_wisdm():
    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)


    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = PReLU()(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = PReLU()(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128)(x)
    x = PReLU()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### **LSTM AND ELU Variation for CNN**


In [ ]:
def build_model_elu_lstm_cnn_wisdm():
    inputs = tf.keras.Input(shape=(44,1))
    lstm_out1 = LSTM(64, return_sequences=True)(inputs)
    lstm_out2 = LSTM(64, return_sequences=True)(lstm_out1)



    conv1 = Conv1D(32, 3, padding='same')(lstm_out2)
    x = tf.keras.activations.elu(conv1)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    conv2 = Conv1D(64, 3, padding='same')(x)
    x = tf.keras.activations.elu(conv2)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2, padding='same')(x)

    x = Flatten()(x)
    x = Dense(128, activation=tf.keras.activations.elu)(x)
    x = Dropout(0.5)(x)
    outputs = Dense(6, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_accuracy',           # Watch validation accuracy
    patience=5,                       # Wait 5 epochs for improvement
    factor=0.5,                       # Halve LR when no progress after activity
    min_lr=1e-7,                      # Minimum LR floor
    verbose=1,
    mode='max'
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1,
    mode='max'
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score
)
model_builders_wisdm = {
    'model_lstm': build_model_lstm_wisdm,
    'model_two_way': build_model_1_wisdm,
    'model_relu': build_model_2_wisdm,
    'model_squareroot_log': build_model_3_wisdm,
    'model_nonlin_taylor_newton_raphson_4': build_model_nonlin_optimized_4_wisdm,
    'model_nonlin_taylor_newton_raphson_3': build_model_nonlin_optimized_3_wisdm,
    'model_stabilized_log': build_model_7_wisdm,
    'model_tanh': build_model_tanh_wisdm,
    'model_crelu': build_model_crelu_wisdm,
    'model_prelu': build_model_prelu_wisdm,
    'model_elu': build_model_elu_wisdm
}
detailed_results_wisdm = []

for model_name, build_model_fn in model_builders_wisdm.items():
    print(f"\nTraining {model_name}")

    model = build_model_fn()

    start_time = time.time()

    history = model.fit(X_train_wisdm, y_train_wisdm,
                       epochs=20,
                       batch_size=64,
                       validation_data=(X_val_wisdm, y_val_wisdm),
                       callbacks=[lr_scheduler, early_stopping],
                       verbose=1)

    elapsed_time = time.time() - start_time

    y_pred_probs = model.predict(X_test_wisdm)
    y_pred_test = np.argmax(y_pred_probs, axis=1)
    y_true_test = np.argmax(y_test_wisdm, axis=1)

    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_f1 = f1_score(y_true_test, y_pred_test, average='weighted')

    test_recall = recall_score(y_true_test, y_pred_test, average='weighted')
    test_precision = precision_score(y_true_test, y_pred_test, average='weighted')
    test_confusion = confusion_matrix(y_true_test, y_pred_test)
    test_auc = roc_auc_score(y_test_wisdm, y_pred_probs, multi_class='ovr', average='weighted')
    test_log_loss = log_loss(y_test_wisdm, y_pred_probs)
    start_inf = tf.timestamp()
    _ = model.predict(X_test_wisdm, batch_size=32)
    end_inf = tf.timestamp()
    test_inference_time = float(end_inf - start_inf)

    detailed_results_wisdm.append({
        'Model': model_name,
        'Test_Accuracy': test_acc,
        'Test_F1_Score': test_f1,
        'Test_Recall_Score': test_recall,
        'Test_Precision_Score': test_precision,
        'Test_Confusion_Matrix': test_confusion,
        'Test_AUC_Score': test_auc,
        'Test_Log_Loss': test_log_loss,
        'Test_Inference_Time_sec': test_inference_time,

        'Train_Time_sec': elapsed_time,
        'Epochs_Trained': len(history.history['loss']),
        'Final_Train_Loss': history.history['loss'][-1],
        'Final_Val_Loss': history.history['val_loss'][-1],
        'Best_Val_Loss': min(history.history['val_loss']),
        'Train_Loss_History': history.history['loss'],
        'Val_Loss_History': history.history['val_loss'],
        'Overfitting_Gap': history.history['loss'][-1] - history.history['val_loss'][-1]
    })


In [ ]:
model_builders_lstm_wisdm = {
    'model_2way_cnn_lstm': build_model_4_wisdm,
    'model_relu_cnn_lstm': build_model_5_wisdm,
    'model_cnn_lstm_sqaure_root': build_model_6_wisdm,
    'model_nonlin_optimized_lstm_cnn_3series': build_model_nonlin_optimized_lstm_cnn_3_wisdm,
    'model_nonlin_optimized_lstm_cnn_4series': build_model_nonlin_optimized_lstm_cnn_4_wisdm,
    'model_stab_log_lstm_cnn': build_model_CNN_LSTM_stablized_log_wisdm,
    'model_tanh_lstm_cnn': build_model_tanh_lstm_cnn_wisdm,
    'model_crelu_lstm_cnn': build_model_crelu_lstm_cnn_wisdm,
    'model_prelu_lstm_cnn': build_model_prelu_lstm_cnn_wisdm,
    'model_elu_lstm_cnn': build_model_elu_lstm_cnn_wisdm
}

test_results_summary_hybrid_wisdm = []

for model_name, build_model_fn in model_builders_lstm_wisdm.items():
    print(f"\nTraining {model_name}")

    model = build_model_fn()

    start_time = time.time()
    history = model.fit(X_train_wisdm, y_train_wisdm,
              epochs=20,
              batch_size=64,
              validation_data=(X_val_wisdm, y_val_wisdm),
              callbacks=[lr_scheduler, early_stopping],
              verbose=1)
    elapsed_time = time.time() - start_time

    y_pred_probs = model.predict(X_test_wisdm)
    y_pred_test = np.argmax(y_pred_probs, axis=1)
    y_true_test = np.argmax(y_test_wisdm, axis=1)

    test_acc = accuracy_score(y_true_test, y_pred_test)
    test_prec = precision_score(y_true_test, y_pred_test, average='weighted')
    test_rec = recall_score(y_true_test, y_pred_test, average='weighted')
    test_f1 = f1_score(y_true_test, y_pred_test, average='weighted')
    test_confusion = confusion_matrix(y_true_test, y_pred_test)
    test_auc = roc_auc_score(y_test_wisdm, y_pred_probs, multi_class='ovr', average='weighted')
    test_log_loss = log_loss(y_test_wisdm, y_pred_probs)
    start_inf = tf.timestamp()
    _ = model.predict(X_test_wisdm, batch_size=32)
    end_inf = tf.timestamp()
    test_inference_time = float(end_inf - start_inf)
    print(f"Test Accuracy for {model_name}: {test_acc:.4f}")
    print(f"Test F1-score for {model_name}: {test_f1:.4f}")

    test_results_summary_hybrid_wisdm.append({
        'Model': model_name,
        'Test_Accuracy': test_acc,
        'Test_Precision': test_prec,
        'Test_Recall': test_rec,
        'confusion_matrix': test_confusion,
        'Test_AUC_Score': test_auc,
        'Test_Log_Loss': test_log_loss,
        'Test_Inference_Time_sec': test_inference_time,
        'Test_F1_Score': test_f1,
        'Train_Time_sec': elapsed_time,
        'Epochs_Trained': len(history.history['loss']),
        'Final_Train_Loss': history.history['loss'][-1],
        'Final_Val_Loss': history.history['val_loss'][-1],
        'Best_Val_Loss': min(history.history['val_loss']),
        'Train_Loss_History': history.history['loss'],
        'Val_Loss_History': history.history['val_loss'],
        'Overfitting_Gap': history.history['loss'][-1] - history.history['val_loss'][-1]
    })


In [ ]:
detailed_results_2_wisdm = pd.DataFrame(final_dataframe_wisdm)
detailed_results_2_wisdm

In [ ]:
detailed_results_2_wisdm = detailed_results_2_wisdm.rename(columns={
    'Test_Recall': 'Test_Recall_Score',
    'Test_Precision': 'Test_Precision_Score',
    'confusion_matrix': 'Test_Confusion_Matrix'
})

final_dataframe_wisdm

In [ ]:
wisdm_results = final_dataframe_wisdm
wisdm_results.to_csv("wisdm.csv", index=False)

### **ANALYSING**

In [ ]:
def confusion_to_class_metrics(cm):
    cm = np.array(cm)

    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    eps = 1e-9

    precision = TP / (TP + FP + eps)
    recall    = TP / (TP + FN + eps)
    f1        = 2 * (precision * recall) / (precision + recall + eps)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


def build_classwise_comparison(df, class_names, metric="f1"):
    comparison = {cls: {} for cls in class_names}

    for _, row in df.iterrows():
        model_name = row["Model"]
        cm = row["Test_Confusion_Matrix"]

        # Skip invalid / missing confusion matrices
        if cm is None:
            continue

        metrics = confusion_to_class_metrics(cm)

        for i, cls in enumerate(class_names):
            comparison[cls][model_name] = metrics[metric][i]

    return pd.DataFrame(comparison).T


def best_model_per_activity(comparison_df):
    return comparison_df.idxmax(axis=1)


# ----------------------------------
# WISDM Class labels (6 activities)
# ----------------------------------

wisdm_class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


comparison_wisdm = build_classwise_comparison(
    final_dataframe_wisdm,
    wisdm_class_names,
    metric="f1"
)


best_models_wisdm = best_model_per_activity(comparison_wisdm).reset_index()
best_models_wisdm.columns = ["Activity", "Best_Model"]


In [ ]:
best_models_wisdm

In [ ]:
cm = final_dataframe_wisdm["Test_Confusion_Matrix"].dropna().iloc[0]
cm = np.array(cm)

# Row-wise sum gives number of true samples per class
wisdm_class_samples = pd.Series(
    cm.sum(axis=1),
    index=comparison_wisdm.index
)

# Optional: percentage support
wisdm_class_support = wisdm_class_samples / wisdm_class_samples.sum() * 100

# Combine into one table
wisdm_class_distribution = pd.DataFrame({
    "Samples": wisdm_class_samples,
    "Support (%)": wisdm_class_support
})

print("\n==============================")
print(" WISDM TEST SET CLASS DISTRIBUTION ")
print("==============================\n")
print(wisdm_class_distribution)


In [ ]:
comparison_analysis_wisdm = (
    best_models_wisdm
    .set_index("Activity")
    .join(wisdm_class_distribution)
)

comparison_analysis_wisdm = comparison_analysis_wisdm.sort_values(
    "Support (%)", ascending=False
)

comparison_analysis_wisdm


In [ ]:
# Copy dataframe (WISDM)
df = comparison_analysis_wisdm.copy()

# Proposed models list (UNCHANGED)
proposed_models = [
    "model_two_way",
    "model_squareroot_log",
    "model_nonlin_taylor_newton_raphson_4",
    "model_nonlin_taylor_newton_raphson_3",
    "model_stabilized_log",
    "model_2way_cnn_lstm",
    "model_cnn_lstm_sqaure_root",
    "model_nonlin_optimized_lstm_cnn_3series",
    "model_nonlin_optimized_lstm_cnn_4series",
    "model_stab_log_lstm_cnn"
]

# Assign colors
shade_A = "#1f77b4"  # proposed models
shade_B = "#A7D4FF"  # existing models

df["Color"] = df["Best_Model"].apply(
    lambda x: shade_A if x in proposed_models else shade_B
)

# Sort by Samples for horizontal bars
df_sorted = df.sort_values("Samples", ascending=True)

# ----------------------------------
# Plot
# ----------------------------------
plt.figure(figsize=(12, 8))
bars = plt.barh(
    df_sorted.index,
    df_sorted["Samples"],
    color=df_sorted["Color"]
)

# Annotate bars
for bar, (_, row) in zip(bars, df_sorted.iterrows()):
    width = bar.get_width()
    label_x_pos = width - 20 if width > 60 else width + 5
    ha = 'right' if width > 60 else 'left'

    plt.text(
        label_x_pos,
        bar.get_y() + bar.get_height() / 2,
        f"{row['Best_Model']} ({row['Support (%)']:.1f}%)",
        va='center',
        ha=ha,
        fontsize=10,
        fontweight='bold'
    )

# Labels and title
plt.xlabel("Number of Samples", fontsize=12, fontweight='bold')
plt.ylabel("Activity", fontsize=12, fontweight='bold')
plt.title(
    "WISDM: Class-wise Best Model\n(Proposed vs Existing Models)",
    fontsize=14,
    fontweight='bold'
)

# Legend
legend_elements = [
    mpatches.Patch(color=shade_A, label="Proposed Models"),
    mpatches.Patch(color=shade_B, label="Existing Models")
]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
from itertools import combinations
from sklearn.preprocessing import MinMaxScaler

# -------------------------------
# Define metrics
# -------------------------------
perf_metrics_wisdm = [
    "Test_Accuracy",
    "Test_F1_Score",
    "Test_Recall_Score",
    "Test_Precision_Score",
    "Test_AUC_Score"
]

cost_metrics_wisdm = [
    "Train_Time_sec",
    "Test_Inference_Time_sec",
    "Test_Log_Loss",
    "Overfitting_Gap"
]

# Use WISDM dataframe
df_wisdm = final_dataframe_wisdm.set_index("Model").copy()

# -------------------------------
# Normalize metrics
# -------------------------------
scaler = MinMaxScaler()

# Performance metrics: higher is better
df_perf_norm_wisdm = pd.DataFrame(
    scaler.fit_transform(df_wisdm[perf_metrics_wisdm]),
    columns=perf_metrics_wisdm,
    index=df_wisdm.index
)

# Cost metrics: lower is better → invert
df_cost_norm_wisdm = pd.DataFrame(
    1 - scaler.fit_transform(df_wisdm[cost_metrics_wisdm]),
    columns=cost_metrics_wisdm,
    index=df_wisdm.index
)

# -------------------------------
# Function to compute UIR (unchanged)
# -------------------------------
def compute_uir(metrics_df):
    model_names = metrics_df.index.tolist()
    uir_matrix = pd.DataFrame(
        np.zeros((len(model_names), len(model_names))),
        index=model_names,
        columns=model_names
    )

    for model_i, model_j in combinations(model_names, 2):
        metrics_i = metrics_df.loc[model_i].values
        metrics_j = metrics_df.loc[model_j].values

        unanimous = metrics_i >= metrics_j
        if unanimous.all():
            uir_matrix.loc[model_i, model_j] = 1
        elif (metrics_j >= metrics_i).all():
            uir_matrix.loc[model_j, model_i] = 1
        else:
            pass  # no domination

    uir_summary = uir_matrix.sum(axis=1).sort_values(ascending=False)
    uir_ranking = uir_summary.rank(ascending=False, method='min')

    return uir_summary, uir_ranking

# -------------------------------
# Compute UIR (WISDM)
# -------------------------------
perf_uir_summary_wisdm, perf_uir_ranking_wisdm = compute_uir(df_perf_norm_wisdm)
cost_uir_summary_wisdm, cost_uir_ranking_wisdm = compute_uir(df_cost_norm_wisdm)

# -------------------------------
# Display results
# -------------------------------
print("\n==============================")
print("WISDM – PERFORMANCE UIR SUMMARY")
print("==============================\n")
print(perf_uir_summary_wisdm)

print("\nWISDM – Performance UIR ranking:\n", perf_uir_ranking_wisdm)

print("\n==============================")
print("WISDM – COST / EFFICIENCY UIR SUMMARY")
print("==============================\n")
print(cost_uir_summary_wisdm)

print("\nWISDM – Cost/Efficiency UIR ranking:\n", cost_uir_ranking_wisdm)


In [ ]:
    # -------------------------------
    # Prepare dataframe for plotting (WISDM)
    # -------------------------------
    df_plot_wisdm = pd.DataFrame({
        "Performance_UIR": perf_uir_summary_wisdm,
        "Cost_UIR": cost_uir_summary_wisdm
    })

    # Sort for better visualization
    df_perf_sorted_wisdm = df_plot_wisdm.sort_values("Performance_UIR", ascending=False)
    df_cost_sorted_wisdm = df_plot_wisdm.sort_values("Cost_UIR", ascending=False)

    # -------------------------------
    # Define proposed models (unchanged list)
    # -------------------------------
    proposed_models = [
        "model_two_way",
        "model_squareroot_log",
        "model_nonlin_taylor_newton_raphson_4",
        "model_nonlin_taylor_newton_raphson_3",
        "model_stabilized_log",
        "model_2way_cnn_lstm",
        "model_cnn_lstm_sqaure_root",
        "model_nonlin_optimized_lstm_cnn_3series",
        "model_nonlin_optimized_lstm_cnn_4series",
        "model_stab_log_lstm_cnn"
    ]

    # -------------------------------
    # Assign colors
    # -------------------------------
    shade_A = "#1f77b4"  # proposed models
    shade_B = "#A7D4FF"  # existing models

    df_perf_sorted_wisdm["Color"] = df_perf_sorted_wisdm.index.to_series().apply(
        lambda x: shade_A if x in proposed_models else shade_B
    )

    df_cost_sorted_wisdm["Color"] = df_cost_sorted_wisdm.index.to_series().apply(
        lambda x: shade_A if x in proposed_models else shade_B
    )

    # -------------------------------
    # Performance UIR - vertical bar chart (WISDM)
    # -------------------------------
    plt.figure(figsize=(18, 8))
    plt.bar(
        df_perf_sorted_wisdm.index,
        df_perf_sorted_wisdm["Performance_UIR"],
        color=df_perf_sorted_wisdm["Color"],
        edgecolor='black'
    )
    plt.xticks(rotation=90, fontsize=10)
    plt.ylabel("Performance UIR", fontsize=12, fontweight='bold')
    plt.title("WISDM – Performance UIR per Model", fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # -------------------------------
    # Cost/Efficiency UIR - vertical bar chart (WISDM)
    # -------------------------------
    plt.figure(figsize=(18, 8))
    plt.bar(
        df_cost_sorted_wisdm.index,
        df_cost_sorted_wisdm["Cost_UIR"],
        color=df_cost_sorted_wisdm["Color"],
        edgecolor='black'
    )
    plt.xticks(rotation=90, fontsize=10)
    plt.ylabel("Cost/Efficiency UIR", fontsize=12, fontweight='bold')
    plt.title("WISDM – Cost/Efficiency UIR per Model", fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


### **PARETO**

In [ ]:
# -------------------------------
# Compute aggregate performance and cost scores for Pareto analysis (WISDM)
# -------------------------------
df_combined_wisdm = pd.DataFrame({
    "Performance_Score": df_perf_norm_wisdm.mean(axis=1),   # average normalized performance
    "Cost_Score": df_cost_norm_wisdm.mean(axis=1)           # average normalized efficiency (higher is better after inversion)
})

# -------------------------------
# Function to compute Pareto front
# -------------------------------
def pareto_front_wisdm(df):
    is_dominated = np.zeros(len(df), dtype=bool)

    for i, row_i in df.iterrows():
        for j, row_j in df.iterrows():
            if i == j:
                continue
            # Check if j dominates i
            if (row_j["Performance_Score"] >= row_i["Performance_Score"]) and \
               (row_j["Cost_Score"] >= row_i["Cost_Score"]) and \
               ((row_j["Performance_Score"] > row_i["Performance_Score"]) or
                (row_j["Cost_Score"] > row_i["Cost_Score"])):

                is_dominated[df.index.get_loc(i)] = True
                break

    return df[~is_dominated]

# -------------------------------
# Compute Pareto-optimal models (WISDM)
# -------------------------------
pareto_models_wisdm = pareto_front_wisdm(df_combined_wisdm)

print("\nPareto-optimal models (WISDM):\n")
print(pareto_models_wisdm)


In [ ]:
# -------------------------------
# Pareto distance ranking (WISDM)
# -------------------------------
pareto_df_wisdm = pd.DataFrame({
    "Performance_Score": df_perf_norm_wisdm.mean(axis=1),  # normalized performance score
    "Cost_Score": df_cost_norm_wisdm.mean(axis=1)          # normalized cost score
})

# Ideal point (both performance and cost/efficiency maxed)
ideal_point = np.array([1, 1])

# Compute Euclidean distance to ideal point for each model
pareto_df_wisdm["Distance_to_Ideal"] = np.sqrt(
    (pareto_df_wisdm["Performance_Score"] - ideal_point[0])**2 +
    (pareto_df_wisdm["Cost_Score"] - ideal_point[1])**2
)

# Rank models: smaller distance = better
pareto_df_wisdm["Rank"] = pareto_df_wisdm["Distance_to_Ideal"].rank(method="min")

# Sort by rank
pareto_df_sorted_wisdm = pareto_df_wisdm.sort_values("Rank")

print("\nModel ranking based on distance to Pareto-optimal point (WISDM):\n")
print(pareto_df_sorted_wisdm)


In [ ]:
from adjustText import adjust_text

df_plot_wisdm = pareto_df_sorted_wisdm.copy()

pareto_mask_wisdm = df_plot_wisdm.index.isin(pareto_models_wisdm.index)
top5_mask_wisdm = df_plot_wisdm.head(5).index

# Highlight only: Pareto-optimal + Top 5 closest (union)
highlight_models_wisdm = df_plot_wisdm.loc[pareto_mask_wisdm | df_plot_wisdm.index.isin(top5_mask_wisdm)]

# -------------------------------
# Plot
# -------------------------------
plt.figure(figsize=(12, 9))
plt.style.use('default')  # Clean look

# 1. Pareto-optimal models
pareto_points_wisdm = highlight_models_wisdm[highlight_models_wisdm.index.isin(pareto_models_wisdm.index)]
plt.scatter(pareto_points_wisdm["Cost_Score"], pareto_points_wisdm["Performance_Score"],
            c='#d62728', s=180, edgecolors='black', linewidth=1.2,
            label='Pareto-optimal', zorder=10)

# 2. Top-3 closest (only if not already Pareto)
top5_only_wisdm = highlight_models_wisdm[highlight_models_wisdm.index.isin(top5_mask_wisdm)]
top5_only_wisdm = top5_only_wisdm[~top5_only_wisdm.index.isin(pareto_models_wisdm.index)]

plt.scatter(top5_only_wisdm["Cost_Score"], top5_only_wisdm["Performance_Score"],
            c='#1f77b4', s=160, edgecolors='black', linewidth=1,
            label='Top-5 closest to ideal', zorder=9)

# 3. Smart labels
texts_wisdm = []
for model_name, row in highlight_models_wisdm.iterrows():
    color = '#d62728' if model_name in pareto_models_wisdm.index else '#1f77b4'
    weight = 'bold' if model_name in pareto_models_wisdm.index else 'normal'
    texts_wisdm.append(
        plt.text(row["Cost_Score"], row["Performance_Score"], f" {model_name}",
                 fontsize=10.5, fontweight=weight, color=color,
                 ha='left', va='center')
    )

adjust_text(texts_wisdm, arrowprops=dict(arrowstyle='-', color='gray', lw=0.8, alpha=0.6))



# -------------------------------
# Styling and zoom axes to top models only
# -------------------------------
margin = 0.02
plt.xlim(highlight_models_wisdm["Cost_Score"].min() - margin, highlight_models_wisdm["Cost_Score"].max() + margin)
plt.ylim(highlight_models_wisdm["Performance_Score"].min() - margin, highlight_models_wisdm["Performance_Score"].max() + margin)

plt.xlabel("Cost/Efficiency Score (Higher → Better)", fontsize=13, fontweight='bold')
plt.ylabel("Performance Score (Higher → Better)", fontsize=13, fontweight='bold')
plt.title("WISDM: Pareto-optimal & Top-5 Closest Models",
          fontsize=16, fontweight='bold', pad=20)

plt.grid(True, alpha=0.3)
plt.legend(loc='lower left', frameon=True, fancybox=True, shadow=True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import rankdata

# ====================== PREP ======================
df = final_dataframe_wisdm.copy()
df = df.reset_index(drop=True)

# Clean names
df['Model_Clean'] = (
    df['Model']
    .str.replace('model_', '', regex=False)
    .str.replace('_', ' ', regex=False)
    .str.title()
    .str.replace('Nonlin', 'Non-Linear')
    .str.replace('Crelu', 'C-ReLU')
    .str.replace('Prelu', 'PReLU')
    .str.replace('Elu', 'ELU')
    .str.replace('Lstm', 'LSTM')
    .str.replace('Two Way', 'Two-Way')
)

sns.set_style("whitegrid")

# =======================================================
# SHADE-A / SHADE-B LOGIC
# =======================================================

ordered_models = df['Model'].tolist()

shade_A_indices = [1, 3, 4, 5, 6, 11, 13, 14, 15, 16]

shade_A = "#1f77b4"
shade_B = "#A7D4FF"

colors = [
    shade_A if i in shade_A_indices else shade_B
    for i in range(len(ordered_models))
]

# ====================== BAR CHARTS — MAIN PERFORMANCE METRICS ======================
metrics_line = [
    ('Test_Accuracy',        'Test Accuracy'),
    ('Test_F1_Score',        'Test F1-Score'),
    ('Test_Precision_Score', 'Test Precision'),
    ('Test_Recall_Score',    'Test Recall'),
    ('Test_AUC_Score',       'Test AUC Score')
]

for col, title in metrics_line:
    plt.figure(figsize=(12, 7))

    plt.bar(
        df['Model_Clean'],
        df[col],
        color=colors,
        edgecolor='black',
        linewidth=1.4
    )

    plt.title(title, fontsize=20, pad=20)
    plt.ylabel(title)
    plt.xticks(rotation=45, ha='right')

    for x, y in zip(df['Model_Clean'], df[col]):
        plt.text(x, y + 0.002, f'{y:.4f}',
                 ha='center', fontsize=10, fontweight='bold')

    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# ====================== BAR CHARTS — COST / EFFICIENCY METRICS ======================
metrics_to_plot = [
    ('Test_Log_Loss',           'Test Log Loss'),
    ('Test_Inference_Time_sec', 'Inference Time (seconds)'),
    ('Train_Time_sec',          'Training Time (seconds)'),
    ('Best_Val_Loss',           'Best Validation Loss'),
    ('Overfitting_Gap',         'Overfitting Gap'),
]

for col, title in metrics_to_plot:
    plt.figure(figsize=(11, 7))

    bars = plt.bar(
        df['Model_Clean'],
        df[col],
        color=colors,
        edgecolor='black',
        linewidth=1.4
    )

    plt.title(title, fontsize=20, pad=20)
    plt.ylabel(title.split('(')[0].strip())
    plt.xticks(rotation=45, ha='right')

    for bar, val in zip(bars, df[col]):
        h = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            h + abs(h)*0.02,
            f'{val:.4f}' if col != 'Test_Inference_Time_sec' else f'{val:.3f}s',
            ha='center',
            va='bottom',
            fontweight='bold'
        )

    plt.tight_layout()
    plt.show()


In [ ]:
final_dataframe_ucihapt